In [1]:
from pathlib import Path

import pandas as pd

In [2]:
sold_path = Path("data/cleaned/CRMLSSold_cleaned.csv")
date_columns = ["CloseDate", "ListingContractDate", "PurchaseContractDate"]

sold = pd.read_csv(sold_path, parse_dates=date_columns, low_memory=False)
sold.shape

(443092, 53)

### Feature engineering

In [3]:
sold["PriceRatio"] = sold["ClosePrice"] / sold["OriginalListPrice"]
sold["CloseToOriginalListRatio"] = sold["PriceRatio"]
sold["PricePerSqFt"] = sold["ClosePrice"] / sold["LivingArea"]

sold["Year"] = sold["CloseDate"].dt.year
sold["Month"] = sold["CloseDate"].dt.month
sold["YrMo"] = sold["CloseDate"].dt.strftime("%Y-%m")

sold["ListingToContractDays"] = (
    sold["PurchaseContractDate"] - sold["ListingContractDate"]
).dt.days
sold["ContractToCloseDays"] = (
    sold["CloseDate"] - sold["PurchaseContractDate"]
).dt.days

### Sample output

In [4]:
# sample_columns = [
#     "ClosePrice", "OriginalListPrice", "LivingArea", "DaysOnMarket",
#     "CloseDate", "ListingContractDate", "PurchaseContractDate",
#     "PriceRatio", "CloseToOriginalListRatio", "PricePerSqFt",
#     "Year", "Month", "YrMo", "ListingToContractDays",
#     "ContractToCloseDays",
# ]
sold

,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,LivingArea,...,purchase_after_close_flag,negative_timeline_flag,PriceRatio,CloseToOriginalListRatio,PricePerSqFt,Year,Month,YrMo,ListingToContractDays,ContractToCloseDays
0,"Carpet,Tile,Wood",True,False,499000.0,2024-01-26,240000.0,37.566106,-122.327954,1 Baldwin Avenue 411,1140.0,...,False,False,0.480962,0.480962,210.526316,2024,1,2024-01,777.0,65.0
1,NaN,False,False,759900.0,2024-01-05,815000.0,32.659315,-117.096922,2811 C Avenue,1974.0,...,False,False,1.072510,1.072510,412.867275,2024,1,2024-01,114.0,919.0
2,NaN,False,False,739900.0,2024-01-05,810000.0,32.659284,-117.097174,2812 C Avenue,1974.0,...,False,False,1.094743,1.094743,410.334347,2024,1,2024-01,255.0,778.0
3,NaN,True,False,2100000.0,2024-01-02,2100000.0,35.277199,-120.646786,2054 Fixlini Street,3736.0,...,False,False,1.000000,1.000000,562.098501,2024,1,2024-01,0.0,48.0
4,"Tile,Wood",True,NaN,NaN,2024-01-31,2340000.0,37.325454,-121.780303,3114 Rasmus Circle,2442.0,...,True,False,NaN,NaN,958.230958,2024,1,2024-01,33.0,-33.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
443087,NaN,True,False,1388000.0,2026-06-30,13000000.0,34.117226,-118.372189,2525 Thames Street,1300.0,...,False,False,9.365994,9.365994,10000.000000,2026,6,2026-06,9.0,867.0
443088,NaN,True,True,919900.0,2026-06-30,850000.0,34.635955,-118.213539,4638 W Avenue M10,3335.0,...,False,False,0.924013,0.924013,254.872564,2026,6,2026-06,189.0,943.0
443089,"Carpet,Laminate",NaN,False,975000.0,2026-06-11,595000.0,37.824500,-122.281570,3228 Magnolia St,2502.0,...,False,False,0.610256,0.610256,237.809752,2026,6,2026-06,961.0,182.0
443090,"Carpet,Laminate,Wood",True,False,649999.0,2026-06-30,615000.0,34.639233,-118.178343,41614 27th Street W,2454.0,...,False,False,0.946155,0.946155,250.611247,2026,6,2026-06,125.0,1244.0


In [5]:
def summary_stat(df, column):
    summary_columns = [
    "ClosePrice", "PricePerSqFt", "DaysOnMarket",
    "CloseToOriginalListRatio",
    ]
    return (df.groupby(column)[summary_columns]
            .agg(["count", "mean", "median"]).round(2))

In [6]:
property_summary = summary_stat(sold, "PropertySubType")
property_summary

ClosePrice                        PricePerSqFt           \
                           count        mean     median        count     mean   
PropertySubType                                                                 
BoatSlip                      51   263025.49   185000.0           51  1971.41   
Cabin                        507   290079.28   241000.0          507   592.38   
CoOwnership                   20   723124.95   405000.0           20   474.34   
Condominium                72534   877251.80   625000.0        72534      inf   
DeededParking                  4   656250.00   577500.0            4   410.51   
Duplex                      2471  1221050.65   910000.0         2471   638.74   
Farm                          15  1771266.67  1575000.0           15   878.05   
Loft                          31   680638.35   650000.0           31   551.08   
ManufacturedHome              47   328291.79   305000.0           47   214.45   
ManufacturedOnLand          5764   351419.54   320750.0         5764      inf   
MixedUse                     221   922791.45   675000.0          221   496.64   
MobileHome                    65   444869.23   495000.0           65      inf   
OwnYourOwn                    66   358624.33   275000.0           66      inf   
Quadruplex                   159  1447787.43  1275000.0          159   447.07   
SingleFamilyResidence     332327  1292273.92   895000.0       332327      inf   
StockCooperative            1752   392446.10   360000.0         1752   407.41   
Studio                        11   350953.64   322000.0           11   692.59   
Timeshare                     18   455861.11   505000.0           18   305.53   
Townhouse                  25825  1008787.24   800000.0        25825   656.70   
Triplex                      379  1317781.90  1135000.0          379   583.39   

                               DaysOnMarket                \
                        median        count   mean median   
PropertySubType                                             
BoatSlip               1750.00           51  50.88   18.0   
Cabin                   287.58          507  79.56   47.0   
CoOwnership             411.65           20  36.40   20.5   
Condominium             563.21        72534  42.05   24.0   
DeededParking           348.41            4  26.00   17.0   
Duplex                  543.33         2471  42.70   21.0   
Farm                    763.36           15  52.40   16.0   
Loft                    539.22           31  63.29   26.0   
ManufacturedHome        190.42           47  62.77   43.0   
ManufacturedOnLand      225.54         5764  59.05   30.0   
MixedUse                433.95          221  84.57   52.0   
MobileHome              756.41           65  81.45   76.0   
OwnYourOwn              508.27           66  65.36   41.0   
Quadruplex              376.95          159  56.30   34.0   
SingleFamilyResidence   532.41       332327  36.14   17.0   
StockCooperative        396.04         1752  38.47   20.5   
Studio                  675.21           11  43.36   18.0   
Timeshare               187.64           18  69.94   58.0   
Townhouse               559.47        25825  32.37   18.0   
Triplex                 473.95          379  55.50   29.0   

                      CloseToOriginalListRatio                 
                                         count    mean median  
PropertySubType                                                
BoatSlip                                    51    0.92   0.95  
Cabin                                      507    0.90   0.92  
CoOwnership                                 20   50.98   0.97  
Condominium                              72435   24.39   0.99  
DeededParking                                4    0.76   0.99  
Duplex                                    2471  394.85   0.98  
Farm                                        14    0.98   0.98  
Loft                                        31    0.99   0.99  
ManufacturedHome                            47    

### County summary

In [7]:
county_summary = summary_stat(sold, "CountyOrParish")
county_summary

ClosePrice                            PricePerSqFt             \
                     count          mean       median        count       mean   
CountyOrParish                                                                  
Alameda              20519  1.316225e+06    1145000.0        20519        inf   
Alpine                   1  1.100000e+06    1100000.0            1     267.12   
Amador                  25  4.132854e+05     410000.0           25     223.85   
Butte                 4522  4.342954e+05     399952.5         4522     261.82   
Calaveras              106  5.440068e+05     475000.0          106     275.30   
Colusa                  28  4.013714e+05     402500.0           28     200.97   
Contra Costa         19807  1.138285e+06     825000.0        19807        inf   
Del Norte                1  2.485000e+06    2485000.0            1     390.91   
El Dorado               83  8.154369e+05     702000.0           83     395.03   
Fresno                1479  4.827207e+05     415000.0         1479        inf   
Glenn                  384  3.655355e+05     350000.0          384     246.37   
Humboldt                13  4.915769e+05     475000.0           13     287.52   
Imperial               314  3.087581e+05     306000.0          314     207.62   
Inyo                     8  4.760625e+05     569250.0            8     323.49   
Kern                  3958  6.217615e+05     370000.0         3958        inf   
Kings                  257  3.766559e+05     355000.0          257     241.27   
Lake                  1933  3.397258e+05     310000.0         1933     218.52   
Lassen                  14  2.748929e+05     131250.0           14     153.76   
Los Angeles         110946  1.317605e+06     905000.0       110946        inf   
Madera                1333  4.745651e+05     425000.0         1333     260.81   
Marin                  150  1.349189e+06    1160000.0          150     754.19   
Mariposa               399  4.750190e+05     405000.0          399     285.61   
Mendocino              470  7.711043e+05     635000.0          470     462.94   
Merced                2778  4.405356e+05     415000.0         2778     264.65   
Modoc                    2  2.820000e+05     282000.0            2     104.86   
Mono                    15  1.032600e+06     915000.0           15     653.31   
Monterey              4112  1.380467e+06     900000.0         4112     733.12   
Napa                   119  1.068460e+06     860000.0          119        inf   
Nevada                  51  6.214534e+05     515000.0           51     355.75   
Orange               50331  1.542350e+06    1180000.0        50331        inf   
Other                    3  7.516667e+05     725000.0            3     429.41   
Other County             2  1.880500e+08  188050000.0            2  179264.02   
Placer                 214  7.340670e+05     640500.0          214     351.57   
Plumas                  45  5.132156e+05     395000.0           45     286.41   
Riverside            62038  7.200129e+05     600000.0        62038        inf   
Sacramento             924  5.091233e+05     472500.0          924     321.51   
San Benito             848  8.409640e+05     776000.0          848     435.11   
San Bernardino       41502  5.984597e+05     533000.0        41502        inf   
San Diego            54281  1.479702e+06     900000.0        54281        inf   
San Francisco          992  1.337754e+06    1198950.0          992        inf   
San Joaquin           1759  6.602336e+05     634000.0         1759     324.30   
San Luis Obispo       6509  1.031352e+06     874900.0         6509        inf   
San Mateo             7520  2.206984e+06    1700000.0         7520    1120.54   
Santa Barbara         2229  1.543732e+06     720000.0         2229        inf   
Santa Clara          18929  1.932904e+06    1600000.0        18929    1069.39   
Santa Cruz            3164  1.351547e+06    1199000.0         3164     826.99   
Shasta                 102  3.768909e+05     36

### MLS Area Summary

In [8]:
mls_area_summary = summary_stat(sold, "MLSAreaMajor")
mls_area_summary

ClosePrice              \
                                                       count        mean   
MLSAreaMajor                                                               
1 - Belmont Shore/Park,Naples,Marina Pac,Bay Hrbr        479  1672800.43   
101 - North Inglewood                                    416   730778.03   
102 - South Inglewood                                    170   808510.44   
103 - Ladera Heights                                     129  1712298.58   
105 - Lennox                                              26   681923.08   
...                                                      ...         ...   
YG50 - North Fork                                         43   444453.49   
YG51 - O'Neals                                             1   550000.00   
YG52 - Wishon                                              4   399500.00   
YG60 - Raymond                                             3   408666.67   
YNEZ - Santa Ynez Valley                                  32  2913856.88   

                                                             PricePerSqFt  \
                                                      median        count   
MLSAreaMajor                                                                
1 - Belmont Shore/Park,Naples,Marina Pac,Bay Hrbr  1400000.0          479   
101 - North Inglewood                               675000.0          416   
102 - South Inglewood                               797450.0          170   
103 - Ladera Heights                               1695000.0          129   
105 - Lennox                                        700000.0           26   
...                                                      ...          ...   
YG50 - North Fork                                   400000.0           43   
YG51 - O'Neals                                      550000.0            1   
YG52 - Wishon                                       366500.0            4   
YG60 - Raymond                                      424000.0            3   
YNEZ - Santa Ynez Valley                           2150000.0           32   

                                                                   \
                                                     mean  median   
MLSAreaMajor                                                        
1 - Belmont Shore/Park,Naples,Marina Pac,Bay Hrbr  881.48  814.14   
101 - North Inglewood                              554.80  505.10   
102 - South Inglewood                              570.62  549.01   
103 - Ladera Heights                               656.90  648.25   
105 - Lennox                                       590.54  537.31   
...                                                   ...     ...   
YG50 - North Fork                                  245.44  249.46   
YG51 - O'Neals                                     276.80  276.80   
YG52 - Wishon                                      326.51  255.65   
YG60 - Raymond                                     225.51  225.53   
YNEZ - Santa Ynez Valley                           936.79  828.56   

                                                  DaysOnMarket                 \
                                                         count    mean median   
MLSAreaMajor                                                                    
1 - Belmont Shore/Park,Naples,Marina Pac,Bay Hrbr          479   44.90   17.0   
101 - North Inglewood                                      416   47.24   29.5   
102 - South Inglewood                                      170   59.12   27.5   
103 - Ladera Heights                                       129   51.17   29.0   
105 - Lennox                                                26   57.96   31.0   
...                                                        ...     ...    ...   
YG50 - North Fork                                           43   86.56   47.0   
YG51 - O'Neals                                               1    6.00    6.0   
YG52 - Wishon                                   

In [9]:
office_summary = summary_stat(sold, "ListOfficeName")
office_summary

ClosePrice                         \
                                       count        mean     median   
ListOfficeName                                                        
#1 FLAT FEE-LIBERTY REALTY                 3   843333.33   900000.0   
#1 Home Sales®                             3   649000.00   700000.0   
*                                          1   565000.00   565000.0   
1 Percent Lists Smart Rate Realty          3   527333.33   405000.0   
1 Percent Lists SoCal                     11  1012758.91   820000.0   
...                                      ...         ...        ...   
smithREgroup                               2  1592500.00  1592500.0   
socalhomeshop Inc.                         4  1205225.00  1215450.0   
ten21 Properties                           1   807000.00   807000.0   
uDream Real Estate                         2   526500.00   526500.0   
zRE Group                                  4  3006250.00  2137500.0   

                                  PricePerSqFt                   DaysOnMarket  \
                                         count     mean   median        count   
ListOfficeName                                                                  
#1 FLAT FEE-LIBERTY REALTY                   3   558.80   507.79            3   
#1 Home Sales®                               3   339.73   347.92            3   
*                                            1   643.51   643.51            1   
1 Percent Lists Smart Rate Realty            3   402.52   312.98            3   
1 Percent Lists SoCal                       11   653.89   691.01           11   
...                                        ...      ...      ...          ...   
smithREgroup                                 2   763.59   763.59            2   
socalhomeshop Inc.                           4   512.34   509.65            4   
ten21 Properties                             1   959.57   959.57            1   
uDream Real Estate                           2   270.55   270.55            2   
zRE Group                                    4  1190.38  1172.13            4   

                                                CloseToOriginalListRatio  \
                                    mean median                    count   
ListOfficeName                                                             
#1 FLAT FEE-LIBERTY REALTY         20.67   24.0                        3   
#1 Home Sales®                     42.00   18.0                        3   
*                                  16.00   16.0                        1   
1 Percent Lists Smart Rate Realty  39.33   24.0                        3   
1 Percent Lists SoCal              16.00    7.0                       11   
...                                  ...    ...                      ...   
smithREgroup                       76.50   76.5                        2   
socalhomeshop Inc.                 34.00   40.5                        4   
ten21 Properties                    3.00    3.0                        1   
uDream Real Estate                 41.00   41.0                        2   
zRE Group                          42.00   17.0                        4   

                                                
                                   mean median  
ListOfficeName                                  
#1 FLAT FEE-LIBERTY REALTY         0.98   0.98  
#1 Home Sales®                     0.97   1.00  
*                                  1.05   1.05  
1 Percent Lists Smart Rate Realty  0.97   1.02  
1 Percent Lists SoCal              1.02   1.00  
...                                 ...    ...  
smithREgroup                       0.86   0.86  
socalhomeshop Inc.                 0.98   0.98  
ten21 Properties                   1.08   1.08  
uDream Real Estate                 1.01   1.01  
zRE Group                          0.99   1.00  

[19082 rows x 12 columns]

In [10]:
buyer_office_summary = summary_stat(sold, "BuyerOfficeName")
buyer_office_summary

ClosePrice                        PricePerSqFt  \
                                  count        mean     median        count   
BuyerOfficeName                                                               
& Company Real Estate                 2   859500.00   859500.0            2   
*                                     4   687500.00   797500.0            4   
, Inc                                 1  2225000.00  2225000.0            1   
01718402                              1   515000.00   515000.0            1   
01719850                              1   800000.00   800000.0            1   
...                                 ...         ...        ...          ...   
uDream Real Estate                    3   506000.00   412000.0            3   
up2date Real Estate                   1   740000.00   740000.0            1   
yesenia@myaxisrealestate.com          1   415000.00   415000.0            1   
zDefault Office                       3  1663333.33  1195000.0            3   
zRE Group                             6   903166.67   927500.0            6   

                                               DaysOnMarket                \
                                 mean   median        count   mean median   
BuyerOfficeName                                                             
& Company Real Estate          462.18   462.18            2  15.00   15.0   
*                              508.57   559.59            4  69.50   67.5   
, Inc                         1115.29  1115.29            1   4.00    4.0   
01718402                       258.15   258.15            1  13.00   13.0   
01719850                       337.55   337.55            1   9.00    9.0   
...                               ...      ...          ...    ...    ...   
uDream Real Estate             345.83   390.68            3   8.33    6.0   
up2date Real Estate            495.65   495.65            1   0.00    0.0   
yesenia@myaxisrealestate.com   268.78   268.78            1  19.00   19.0   
zDefault Office                830.45  1043.07            3  25.00   22.0   
zRE Group                      503.34   559.69            6  29.50   28.5   

                             CloseToOriginalListRatio               
                                                count  mean median  
BuyerOfficeName                                                     
& Company Real Estate                               2  0.96   0.96  
*                                                   4  0.99   0.98  
, Inc                                               1  1.06   1.06  
01718402                                            1  1.05   1.05  
01719850                                            1  1.02   1.02  
...                                               ...   ...    ...  
uDream Real Estate                                  3  1.05   1.02  
up2date Real Estate                                 1  0.93   0.93  
yesenia@myaxisrealestate.com                        1  1.06   1.06  
zDefault Office                                     3  1.00   1.01  
zRE Group                                           6  1.00   0.99  

[21780 rows x 12 columns]

In [11]:
sold

,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,LivingArea,...,purchase_after_close_flag,negative_timeline_flag,PriceRatio,CloseToOriginalListRatio,PricePerSqFt,Year,Month,YrMo,ListingToContractDays,ContractToCloseDays
0,"Carpet,Tile,Wood",True,False,499000.0,2024-01-26,240000.0,37.566106,-122.327954,1 Baldwin Avenue 411,1140.0,...,False,False,0.480962,0.480962,210.526316,2024,1,2024-01,777.0,65.0
1,NaN,False,False,759900.0,2024-01-05,815000.0,32.659315,-117.096922,2811 C Avenue,1974.0,...,False,False,1.072510,1.072510,412.867275,2024,1,2024-01,114.0,919.0
2,NaN,False,False,739900.0,2024-01-05,810000.0,32.659284,-117.097174,2812 C Avenue,1974.0,...,False,False,1.094743,1.094743,410.334347,2024,1,2024-01,255.0,778.0
3,NaN,True,False,2100000.0,2024-01-02,2100000.0,35.277199,-120.646786,2054 Fixlini Street,3736.0,...,False,False,1.000000,1.000000,562.098501,2024,1,2024-01,0.0,48.0
4,"Tile,Wood",True,NaN,NaN,2024-01-31,2340000.0,37.325454,-121.780303,3114 Rasmus Circle,2442.0,...,True,False,NaN,NaN,958.230958,2024,1,2024-01,33.0,-33.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
443087,NaN,True,False,1388000.0,2026-06-30,13000000.0,34.117226,-118.372189,2525 Thames Street,1300.0,...,False,False,9.365994,9.365994,10000.000000,2026,6,2026-06,9.0,867.0
443088,NaN,True,True,919900.0,2026-06-30,850000.0,34.635955,-118.213539,4638 W Avenue M10,3335.0,...,False,False,0.924013,0.924013,254.872564,2026,6,2026-06,189.0,943.0
443089,"Carpet,Laminate",NaN,False,975000.0,2026-06-11,595000.0,37.824500,-122.281570,3228 Magnolia St,2502.0,...,False,False,0.610256,0.610256,237.809752,2026,6,2026-06,961.0,182.0
443090,"Carpet,Laminate,Wood",True,False,649999.0,2026-06-30,615000.0,34.639233,-118.178343,41614 27th Street W,2454.0,...,False,False,0.946155,0.946155,250.611247,2026,6,2026-06,125.0,1244.0
